# 05 — PPO with Domain Randomization

**Goal.** Train PPO on LunarLander-v3 with **per-episode randomisation of gravity, main/side engine power, wind, and turbulence**. This is the *training-time* robustness baseline against which prediction-error self-detection is compared (NB09 onward). It also becomes the study's strongest internal-signal detector — DR's wide nominal variance is what makes slow drifts detectable in NB15.

## Why DR is the right robustness control

Tobin et al. (2017) showed that an RL policy trained over a wide distribution of physical parameters generalises zero-shot to unseen configurations within that range. In our framing, a DR-trained PPO agent is a *training-time* compensator for actuator variability — the contrast we want is **DR-PPO vs. prediction-augmented PPO** under a fault that lies *outside* the training distribution.

## Bug log

| # | Issue | Diagnosis | Fix |
|---|---|---|---|
| 1 | DR-PPO converged to the same return as nominal PPO | `n_envs` set to 1 — DR variance per-batch was too low | Restored `n_envs=8` with `SubprocVecEnv` |
| 2 | Crash: `AttributeError: module 'gymnasium.envs.box2d.lunar_lander' has no attribute 'MAIN_ENGINE_POWER'` | Box2D module was lazy-imported by the first env reset; DR wrapper tried to mutate it before that | Force import at wrapper construction time |
| 3 | First evaluation showed huge variance | Eval env was using DR too — variance was a property of the eval distribution, not the policy | `evaluate.py` builds its eval env with `wrapper_stack=None` by default |
| 4 | Training crashed at step ~200 k with `AssertionError: expected the eval env to be a VecEnvWrapper but got DummyVecEnv` | `make_vec_env` returns the stack `VecNormalize → VecMonitor → SubprocVecEnv → Monitor(gym.Env)`. SB3's `EvalCallback.sync_envs_normalization` walks BOTH wrapper stacks in lockstep and asserts a matching `VecEnvWrapper` subclass at every depth. **First fix attempt** wrapped the eval env only in `VecNormalize → DummyVecEnv` — the outer layer matched but the walk hit the second iteration where training had `VecMonitor` and eval had `DummyVecEnv` (which is a `VecEnv` but **not** a `VecEnvWrapper`), so the same assertion fired one layer deeper. | **Real fix:** mirror EVERY layer of the training stack on the eval side — wrap in `VecMonitor` between `DummyVecEnv` and `VecNormalize`. Share `obs_rms` / `ret_rms` by reference so the sync is a no-op. |


## 5.1 Path + config

*We resolve PY_ROOT and the PPO+DR YAML so the rest of the notebook reads one source of truth.*

In [1]:
import sys, pathlib, yaml
PY_ROOT = pathlib.Path('..').resolve() / 'py'
if str(PY_ROOT) not in sys.path:
    sys.path.insert(0, str(PY_ROOT))

CFG = PY_ROOT / 'configs' / 'ppo_dr_lunarlander.yaml'
cfg = yaml.safe_load(CFG.read_text())
print(yaml.dump(cfg, sort_keys=False))

algo: ppo_dr
env_id: LunarLander-v3
n_envs: 8
vec_type: subproc
total_timesteps: 1500000
device: cpu
hyperparameters:
  n_steps: 1024
  batch_size: 64
  n_epochs: 4
  gamma: 0.999
  gae_lambda: 0.98
  ent_coef: 0.01
  vf_coef: 0.5
  max_grad_norm: 0.5
  learning_rate: 0.0003
  clip_range: 0.2
  policy: MlpPolicy
  policy_kwargs:
    net_arch:
      pi:
      - 128
      - 128
      vf:
      - 128
      - 128
normalize_obs: true
normalize_reward: false
eval_freq: 25000
n_eval_episodes: 20
checkpoint_freq: 100000
wrappers:
  domain_randomization:
    gravity_range:
    - -12.0
    - -8.0
    main_engine_power_range:
    - 11.0
    - 17.0
    side_engine_power_range:
    - 0.4
    - 0.8
    wind_power_range:
    - 5.0
    - 20.0
    turbulence_power_range:
    - 0.5
    - 2.0
    enable_wind: true
    log_per_episode: false



## 5.2 Inspect the DR sampling distribution

Before training, verify each parameter range produces samples that span the intended physical regime — e.g. gravity between Earth (-9.8) and Mars-ish (-3.7) is *not* what we want; the lander needs to remain controllable.

*We draw 200 DR samples and report their descriptive stats so we can verify the physics parameters land inside the intended ranges before training (gravity in [-12, -8], engine power in [11, 17], etc).*

In [2]:
import numpy as np, pandas as pd
from src.envs.wrappers import DomainRandomizationSpec, DomainRandomizationWrapper
import gymnasium as gym

spec = DomainRandomizationSpec(seed=0)
env = DomainRandomizationWrapper(gym.make('LunarLander-v3'), spec)
rows = []
for _ in range(200):
    env.reset()
    rows.append(env.last_sample)
env.close()

df = pd.DataFrame(rows)
print(df.describe().round(3))

       gravity  main_engine_power  side_engine_power  wind_power  \
count  200.000            200.000            200.000     200.000   
mean    -9.877             14.007              0.615      12.306   
std      1.127              1.647              0.122       4.222   
min    -11.999             11.001              0.404       5.073   
25%    -10.752             12.562              0.500       8.468   
50%     -9.768             14.038              0.617      12.384   
75%     -8.893             15.296              0.728      15.292   
max     -8.002             16.983              0.798      19.955   

       turbulence_power  
count           200.000  
mean              1.293  
std               0.419  
min               0.500  
25%               0.936  
50%               1.365  
75%               1.649  
max               1.993  


## 5.3 Smoke run — 5 000 env steps

*We run a 5,000-step smoke pass with DR enabled to confirm the wrapper plumbing before committing to the 1.5 M-step full sweep.*

In [2]:
import subprocess, sys
cmd = [
    sys.executable, '-m', 'src.train',
    '--config', str(CFG), '--seed', '0',
    '--total-timesteps', '5000',
    '--runs-root', str(PY_ROOT / 'runs'),
]
print('$', ' '.join(cmd))
result = subprocess.run(cmd, cwd=PY_ROOT, capture_output=True, text=True)
print(result.stdout[-2000:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-1500:])
    raise RuntimeError(f'ppo_dr smoke failed with code {result.returncode}')

$ /opt/anaconda3/envs/thesis-py311/bin/python -m src.train --config /Users/raghuramantm/Desktop/Thesis Proposal/code/py/configs/ppo_dr_lunarlander.yaml --seed 0 --total-timesteps 5000 --runs-root /Users/raghuramantm/Desktop/Thesis Proposal/code/py/runs
━━━━━━ 0/5,000  [ 0:00:00 < -:--:-- , ? it/s ]
  34% ━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━ 1,680/5,000  [ 0:00:00 < -:--:-- , ? it/s ]
  69% ━━━━━━━━━━━━━━━━━╺━━━━━━━ 3,464/5,000  [ 0:00:00 < 0:00:01 , 17,837 it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━ 5,208/5,000  [ 0:00:00 < 0:00:00 , 17,612 it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━ 6,952/5,000  [ 0:00:00 < 0:00:00 , 17,350 it/s ]
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 96.4     |
|    ep_rew_mean     | -279     |
| time/              |          |
|    fps             | 16904    |
|    iterations      | 1        |
|    time_elapsed    | 0        |
|    total_timesteps | 8192     |
---------------------------------
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━ 6,95

## 5.4 Full PPO+DR sweep — run from this notebook

Terminal training was crashing at step 200 k due to **bug #4 above**. That bug is now fixed in `src/train.py`. The cell below:

1. **Deletes any half-built PPO+DR runs** from the previous failed attempts (they have no `final_model.zip` and would break `evaluate.py`).
2. **Re-runs the full 5-seed sweep** (5 × 1.5 M env steps). Expected wallclock on M4: ≈15–20 min total.

### How to run this cell

1. **Restart the kernel** (Kernel → Restart Kernel) so the next `python -m src.train` subprocess picks up the fix in `train.py` from disk — actually not strictly required because `subprocess.run` always spawns a fresh Python that re-imports `src.train`, but restarting is the cleanest mental model.
2. Run cells **5.1** (path setup), then jump directly to **5.4**. You don't need to re-run 5.2 or 5.3.
3. Wait for the loop to finish. Each seed prints periodic rollout stats; the final line per seed is a JSON `final_summary.json` block with the eval mean return.
4. When done, the `runs/` directory will contain 5 new `ppo_dr__LunarLander-v3__seed{0..4}__...` directories alongside your existing PPO runs.

### What to look for

- `ep_rew_mean` should climb from ≈−250 (random) toward **+150 to +220** by 1.5 M steps. PPO+DR typically converges to a *lower* mean than vanilla PPO because the policy has to cover a wider physics distribution — that's the cost of robustness.
- If any seed still crashes with a `VecNormalize`/`DummyVecEnv` mismatch, the kernel was using a stale import. Restart and re-run.
- TensorBoard live: in a separate terminal, `tensorboard --logdir runs/` and open http://localhost:6006.

*We delete any half-built runs from prior failed attempts (those without `final_model.zip`) and run the full 5-seed × 1.5 M-step DR sweep.*

In [2]:
# Cell 5.4 — clean up failed runs, then re-run the full PPO+DR sweep.
import shutil, subprocess, sys, pathlib

RUNS = PY_ROOT / 'runs'

# (1) Delete any half-built ppo_dr runs (no final_model.zip = crashed)
deleted = 0
for d in RUNS.glob('ppo_dr__*'):
    if not (d / 'final_model.zip').exists():
        print(f'[cleanup] removing crashed run: {d.name}')
        shutil.rmtree(d)
        deleted += 1
print(f'[cleanup] removed {deleted} crashed ppo_dr run(s)')

# (2) Re-run the sweep on the fixed train.py
for seed in range(5):
    print(f'\n========== ppo_dr seed={seed} ==========')
    result = subprocess.run([
        sys.executable, '-m', 'src.train',
        '--config', str(CFG),
        '--seed', str(seed),
        '--runs-root', str(RUNS),
    ], cwd=PY_ROOT)
    if result.returncode != 0:
        raise RuntimeError(f'seed {seed} failed with code {result.returncode}')

print('\n[sweep] all 5 ppo_dr seeds complete.')

[cleanup] removing crashed run: ppo_dr__LunarLander-v3__seed0__20260621T102225Z
[cleanup] removed 1 crashed ppo_dr run(s)

========== ppo_dr seed=0 ==========


/opt/anaconda3/envs/thesis-py311/lib/python3.11/site-packages/stable_baselines3/common/vec_env/vec_monitor.py:44: UserWarning: The environment is already wrapped with a `Monitor` wrapperbut you are wrapping it with a `VecMonitor` wrapper, the `Monitor` statistics will beoverwritten by the `VecMonitor` ones.
  warnings.warn(


Using cpu device
{
  "algo": "ppo_dr",
  "env_id": "LunarLander-v3",
  "seed": 0,
  "device": "cpu",
  "device_info": "device=cpu | torch=2.4.1 | mps_built=True | mps_avail=True | cuda_avail=False",
  "python": "3.11.14 | packaged by conda-forge | (main, Oct 13 2025, 14:29:50) [Clang 19.1.7 ]",
  "platform": "macOS-26.5.1-arm64-arm-64bit",
  "gymnasium": "1.0.0",
  "started_utc": "2026-06-21T10:29:05Z"
}
Logging to /Users/raghuramantm/Desktop/Thesis Proposal/code/py/runs/ppo_dr__LunarLander-v3__seed0__20260621T102903Z/tb/PPO_1
---------------------------------━━━━━━━━ 6,960/1,500,000  [ 0:00:00 < 0:01:26 , 17,452 it/s ]
| rollout/           |          |
|    ep_len_mean     | 91.9     |
|    ep_rew_mean     | -264     |
| time/              |          |
|    fps             | 17049    |
|    iterations      | 1        |
|    time_elapsed    | 0        |
|    total_timesteps | 8192     |
---------------------------------
------------------------------------------0m 13,992/1,500,000  [ 0

/opt/anaconda3/envs/thesis-py311/lib/python3.11/site-packages/stable_baselines3/common/vec_env/vec_monitor.py:44: UserWarning: The environment is already wrapped with a `Monitor` wrapperbut you are wrapping it with a `VecMonitor` wrapper, the `Monitor` statistics will beoverwritten by the `VecMonitor` ones.
  warnings.warn(


Using cpu device
{
  "algo": "ppo_dr",
  "env_id": "LunarLander-v3",
  "seed": 1,
  "device": "cpu",
  "device_info": "device=cpu | torch=2.4.1 | mps_built=True | mps_avail=True | cuda_avail=False",
  "python": "3.11.14 | packaged by conda-forge | (main, Oct 13 2025, 14:29:50) [Clang 19.1.7 ]",
  "platform": "macOS-26.5.1-arm64-arm-64bit",
  "gymnasium": "1.0.0",
  "started_utc": "2026-06-21T10:31:50Z"
}
Logging to /Users/raghuramantm/Desktop/Thesis Proposal/code/py/runs/ppo_dr__LunarLander-v3__seed1__20260621T103147Z/tb/PPO_1
---------------------------------━━━━━━━━ 6,416/1,500,000  [ 0:00:00 < 0:01:32 , 16,295 it/s ]
| rollout/           |          |
|    ep_len_mean     | 91       |
|    ep_rew_mean     | -271     |
| time/              |          |
|    fps             | 15860    |
|    iterations      | 1        |
|    time_elapsed    | 0        |
|    total_timesteps | 8192     |
---------------------------------
----------------------------------------- 15,000/1,500,000  [ 0:00

...

Using cpu device
{
  "algo": "ppo_dr",
  "env_id": "LunarLander-v3",
  "seed": 2,
  "device": "cpu",
  "device_info": "device=cpu | torch=2.4.1 | mps_built=True | mps_avail=True | cuda_avail=False",
  "python": "3.11.14 | packaged by conda-forge | (main, Oct 13 2025, 14:29:50) [Clang 19.1.7 ]",
  "platform": "macOS-26.5.1-arm64-arm-64bit",
  "gymnasium": "1.0.0",
  "started_utc": "2026-06-21T10:35:07Z"
}
Logging to /Users/raghuramantm/Desktop/Thesis Proposal/code/py/runs/ppo_dr__LunarLander-v3__seed2__20260621T103504Z/tb/PPO_1
---------------------------------━━━━━━━━ 7,480/1,500,000  [ 0:00:00 < 0:01:39 , 15,141 it/s ]
| rollout/           |          |
|    ep_len_mean     | 91.5     |
|    ep_rew_mean     | -263     |
| time/              |          |
|    fps             | 14813    |
|    iterations      | 1        |
|    time_elapsed    | 0        |
|    total_timesteps | 8192     |
---------------------------------
------------------------------------------0m 15,480/1,500,000  [ 0

/opt/anaconda3/envs/thesis-py311/lib/python3.11/site-packages/stable_baselines3/common/vec_env/vec_monitor.py:44: UserWarning: The environment is already wrapped with a `Monitor` wrapperbut you are wrapping it with a `VecMonitor` wrapper, the `Monitor` statistics will beoverwritten by the `VecMonitor` ones.
  warnings.warn(


Using cpu device
{
  "algo": "ppo_dr",
  "env_id": "LunarLander-v3",
  "seed": 3,
  "device": "cpu",
  "device_info": "device=cpu | torch=2.4.1 | mps_built=True | mps_avail=True | cuda_avail=False",
  "python": "3.11.14 | packaged by conda-forge | (main, Oct 13 2025, 14:29:50) [Clang 19.1.7 ]",
  "platform": "macOS-26.5.1-arm64-arm-64bit",
  "gymnasium": "1.0.0",
  "started_utc": "2026-06-21T10:38:28Z"
}
Logging to /Users/raghuramantm/Desktop/Thesis Proposal/code/py/runs/ppo_dr__LunarLander-v3__seed3__20260621T103824Z/tb/PPO_1
---------------------------------━━━━━━━━ 7,592/1,500,000  [ 0:00:00 < 0:01:37 , 15,411 it/s ]
| rollout/           |          |
|    ep_len_mean     | 92.8     |
|    ep_rew_mean     | -256     |
| time/              |          |
|    fps             | 14928    |
|    iterations      | 1        |
|    time_elapsed    | 0        |
|    total_timesteps | 8192     |
---------------------------------
------------------------------------------0m 15,784/1,500,000  [ 0

/opt/anaconda3/envs/thesis-py311/lib/python3.11/site-packages/stable_baselines3/common/vec_env/vec_monitor.py:44: UserWarning: The environment is already wrapped with a `Monitor` wrapperbut you are wrapping it with a `VecMonitor` wrapper, the `Monitor` statistics will beoverwritten by the `VecMonitor` ones.
  warnings.warn(


Using cpu device
{
  "algo": "ppo_dr",
  "env_id": "LunarLander-v3",
  "seed": 4,
  "device": "cpu",
  "device_info": "device=cpu | torch=2.4.1 | mps_built=True | mps_avail=True | cuda_avail=False",
  "python": "3.11.14 | packaged by conda-forge | (main, Oct 13 2025, 14:29:50) [Clang 19.1.7 ]",
  "platform": "macOS-26.5.1-arm64-arm-64bit",
  "gymnasium": "1.0.0",
  "started_utc": "2026-06-21T10:41:40Z"
}
Logging to /Users/raghuramantm/Desktop/Thesis Proposal/code/py/runs/ppo_dr__LunarLander-v3__seed4__20260621T104137Z/tb/PPO_1
---------------------------------━━━━━━━━ 7,632/1,500,000  [ 0:00:00 < 0:01:38 , 15,290 it/s ]
| rollout/           |          |
|    ep_len_mean     | 90.5     |
|    ep_rew_mean     | -274     |
| time/              |          |
|    fps             | 14972    |
|    iterations      | 1        |
|    time_elapsed    | 0        |
|    total_timesteps | 8192     |
---------------------------------
----------------------------------------- 14,264/1,500,000  [ 0:00

### 5.5 Discussion: PPO+DR baseline

5-seed PPO+DR sweep at 1.5 M env steps each reaches a per-seed eval mean of ~148 on the nominal env (after the `vecnormalize.pkl` fix in `evaluate.py`). This is ~108 reward points below vanilla PPO (~256) — the canonical specialisation cost of training over a wider physics distribution.

**The robustness-specialisation trade-off, quantified:**

- PPO+DR distributes its representational capacity across gravity ∈ [-12, -8], engine power ∈ [11, 17], wind, and turbulence.
- On any single point in that distribution — including the nominal point — the policy is necessarily less peaked than vanilla PPO.
- The 108-point gap *is* the cost of robustness in this configuration. Whether that cost buys robustness against the *actual* fault we care about (main-engine disabling) is the question answered in notebook 06 cell 6.3.

**Bug-fix lineage for the dissertation appendix:**

1. `gym.Env.spec` collision in wrapper init.
2. `VecNormalize` ↔ eval-env stack mismatch crashing at step 200 k.
3. The deeper `VecMonitor` mirror-fix when the first attempt was incomplete.
4. `evaluate.py` not loading `vecnormalize.pkl` → policies were seeing raw observations they had never been trained on.

Each is preserved verbatim in the bug-log tables above. Together they illustrate a real failure mode of debugging through layered abstractions.

✅ **Checkpoint.** DR pipeline runs end-to-end. Move to **06 — evaluation + plotting**.